# Validation notebook for cosmic rays

In [2]:
import os
import math
import numpy as np
import matplotlib.pyplot as plt
from skimage import morphology

from scipy.special import erf
from scipy.stats import poisson, norm
import scipy.stats as stats

from platosim.simfile import SimFile
from platosim.simulation import Simulation
from platosim.validation import switchOffAllEffects

In [3]:
sim = Simulation("Cosmics")
switchOffAllEffects(sim)
sim.outputDir = os.environ["PLATO_WORKDIR"]

# Multiple full-frame exposure

numExposures = 1000
sim["ObservingParameters/NumExposures"] = numExposures
sim["SubField/NumRows"] = 4510
sim["SubField/NumColumns"] = 4510

# With cosmics

print("With cosmics")
sim["Sky/IncludeCosmicsInSubField"] = "yes"
outputWithCosmics = sim.run(removeOutputFile = True)

# Without cosmics

print("Without cosmics")
sim["Sky/IncludeCosmicsInSubField"] = "no"
outputWithoutCosmics = sim.run(removeOutputFile = True)

With cosmics


Without cosmics

terminate called after throwing an instance of 'H5::DataSetIException'



Exception: Simulation.run(): PlatoSim returned with exit code -6.

## Number of "islands"

In [ ]:
numIslands1 = np.array([])
numIslands2 = np.array([])

for exposure in range(numExposures):
    
    print(exposure)
    
    diff = outputWithCosmics.getImage(exposure) - outputWithoutCosmics.getImage(exposure)
    diff[diff!=0] = 1
    
    labels, num = morphology.label(diff, connectivity=1, return_num=True)
    numIslands1 = np.append(numIslands1, num)
    
    labels, num = morphology.label(diff, connectivity=2, return_num=True)
    numIslands2 = np.append(numIslands2, num)

In [ ]:
cycleTime = sim["ObservingParameters/CycleTime"]
print(numIslands2 / cycleTime / numPixels  / (pixelSize**2))

In [ ]:
fig = plt.figure(figsize = (15, 10))
ax = fig.add_subplot(1, 1, 1)



###############
# From PlatoSim
############### 

plt.plot(numIslands2, "b", label="From PlatoSim")
# plt.plot(numIslands1, "b--", label="Number of islands (connectivitiy = 1)")
# plt.plot(numIslands2, "b", label="Number of islands (connectivitiy = 2)")



########
# Theory
########

cosmicHitRate = sim["Sky/Cosmics/CosmicHitRate"]
cycleTime = sim["ObservingParameters/CycleTime"]
numPixels = sim["CCD/NumRows"] * sim["CCD/NumColumns"]
pixelSize = sim["CCD/PixelSize"] / 10000.0

expectedNumCosmics = cosmicHitRate * cycleTime * numPixels * pixelSize**2
plt.plot([0, numExposures], [expectedNumCosmics, expectedNumCosmics], "r", label="Expected number of cosmic hits")



########
# Layout
########

plt.title("Cosmic hits", fontsize = 24)
plt.legend(loc='best', fontsize = 20)

for tick in ax.xaxis.get_major_ticks():
    tick.label.set_fontsize(16)
    
for tick in ax.yaxis.get_major_ticks():
    tick.label.set_fontsize(16)
    
plt.xlabel("Exposure", fontsize = 20)
plt.ylabel("Number of islands", fontsize = 20)

plt.xlim([0, numExposures])

print(np.mean(numIslands1))
print(np.mean(numIslands2))
print(int(expectedNumCosmics))

## Cosmic hit rate

In [ ]:
fig = plt.figure(figsize = (15, 10))
ax = fig.add_subplot(1, 1, 1)


###############
# From PlatoSim
###############

factor = cycleTime * numPixels * pixelSize**2
# plt.hist(numIslands1 / factor, density=True, color="b", alpha=0.5)
plt.hist(numIslands2 / factor, density=True, color="b", label="From PlatoSim")



######################
# Poisson distribution
######################

mu = cosmicHitRate
x = np.arange(poisson.ppf(0.01, mu), poisson.ppf(0.99, mu))
ax.plot(x, poisson.pmf(x, mu), "g", label="Poisson distribution ($\\lambda$ = CHR)")

plt.axvline(x = mu, color = "r", linestyle = "dashed", label = "Expected cosmic hit rate")



########
# Layout
########

plt.title("Cosmic hits", fontsize = 24)
plt.xlabel("Cosmic hit rate [# events / s / cm$^2$]", fontsize = 20)
plt.ylabel("Fraction of exposures", fontsize = 20)

plt.legend(loc='best', fontsize = 20)

for tick in ax.xaxis.get_major_ticks():
    tick.label.set_fontsize(16)
    
for tick in ax.yaxis.get_major_ticks():
    tick.label.set_fontsize(16)
    
# plt.xlim([expectedCosmicHitRate - 10, expectedCosmicHitRate + 10])

## Trail length

In [ ]:
trailLength = np.array([])

for exposure in range(numExposures):
    
    print(exposure)
    
    diff = outputWithCosmics.getImage(exposure) - outputWithoutCosmics.getImage(exposure)
    diff[diff!=0] = 1
    
    trailLength = np.append(trailLength, np.sum(diff))

In [ ]:
fig = plt.figure(figsize = (15, 10))
ax = fig.add_subplot(1, 1, 1)



###############
# From PlatoSim
############### 

# plt.plot(trailLength / expectedNumCosmics, "b", label="From PlatoSim")
# plt.plot(trailLength / numIslands2, "b", label="From PlatoSim")
plt.plot(trailLength / numIslands2 / math.sqrt(2), "b", label="From PlatoSim")
print(np.mean(trailLength / numIslands2 / math.sqrt(2)))



########
# Theory
########

trailLengthInterval = sim["Sky/Cosmics/TrailLength"]

plt.axhline(y = (trailLengthInterval[0] + trailLengthInterval[1]) / 2, color = "r", linestyle = "dashed", label="Expected average trail length")



########
# Layout
########

plt.title("Cosmic hits", fontsize = 24)
plt.legend(loc='best', fontsize = 20)

for tick in ax.xaxis.get_major_ticks():
    tick.label.set_fontsize(16)
    
for tick in ax.yaxis.get_major_ticks():
    tick.label.set_fontsize(16)
    
plt.xlabel("Exposure", fontsize = 20)
plt.ylabel("Average trail length [pixels]", fontsize = 20)

plt.xlim([0, numExposures])
plt.ylim(trailLengthInterval)
plt.ylim([6.5,8.5])

## Intensity

In [ ]:
intensity = np.array([])

for exposure in range(numExposures):
    
    print(exposure)
    
    diff = outputWithCosmics.getImage(exposure) - outputWithoutCosmics.getImage(exposure)
    
    intensity = np.append(intensity, np.sum(diff))

In [ ]:
fig = plt.figure(figsize = (15, 10))
ax = fig.add_subplot(1, 1, 1)



###############
# From PlatoSim
############### 

plt.plot(intensity / numIslands2, "b", label="From PlatoSim")
print(np.mean(intensity / numIslands2))



########
# Theory
########

intensityInterval = sim["Sky/Cosmics/Intensity"]

plt.axhline(y = (intensityInterval[0] + intensityInterval[1]) / 2, color = "r", linestyle = "dashed", label="Expected average intensity")
# plt.axhline(y = intensityInterval[0], color = "r", linestyle = "dashed")
# plt.axhline(y = intensityInterval[1], color = "r", linestyle = "dashed")



########
# Layout
########

plt.title("Cosmic hits", fontsize = 24)
plt.legend(loc='best', fontsize = 20)

for tick in ax.xaxis.get_major_ticks():
    tick.label.set_fontsize(16)
    
for tick in ax.yaxis.get_major_ticks():
    tick.label.set_fontsize(16)
    
plt.xlabel("Exposure", fontsize = 20)
plt.ylabel("Average intensity [e-]", fontsize = 20)

plt.xlim([0, numExposures])
plt.ylim([20000, 25000])

## Entry position

In [ ]:
diff = outputWithCosmics.getImage(0) - outputWithoutCosmics.getImage(0)
diff[diff!=0] = 1

In [ ]:
positions = np.where(diff == 1)

In [ ]:
fig = plt.figure(figsize = (20, 7))
fig.suptitle("Cosmic hits", fontsize=24)
plt.tight_layout()


###########
# Entry row
###########

ax1 = fig.add_subplot(1, 2, 1)

numRows = sim["CCD/NumRows"]
plt.hist(positions[0], density=True, color="b", label="From PlatoSim")

plt.axhline(y = 1 / numRows, color = "r", linestyle = "dashed", label="Expected average trail length")

for tick in ax1.xaxis.get_major_ticks():
    tick.label.set_fontsize(16)
    
for tick in ax1.yaxis.get_major_ticks():
    tick.label.set_fontsize(16)
    
plt.xlabel("Entry row", fontsize = 20)
plt.ylabel("Fraction of cosmic hits", fontsize = 20)

plt.xlim([0, numRows])
plt.ylim([1 / numRows / 2, 1.2 / numRows])

plt.title("Entry row", fontsize=20)



##############
# Entry column
##############

ax2 = fig.add_subplot(1, 2, 2, sharey=ax1)

numColumns = sim["CCD/NumColumns"]
plt.hist(positions[1], density=True, color="b", label="From PlatoSim")

plt.axhline(y = 1 / 4510, color = "r", linestyle = "dashed", label="Expected average trail length")

for tick in ax2.xaxis.get_major_ticks():
    tick.label.set_fontsize(16)
    
for tick in ax.yaxis.get_major_ticks():
    tick.label.set_fontsize(16)
    
plt.xlabel("Entry column", fontsize = 20)

plt.xlim([0, numColumns])

plt.title("Entry column", fontsize=20)

plt.setp(ax2.get_yticklabels(), visible=False)

# Visualisation of the cosmic hits

In [ ]:
fig = plt.figure(figsize = (15, 10))
ax = fig.add_subplot(1, 1, 1)

im = plt.imshow(diff, origin='lower', cmap='Greys')



########
# Layout
########

plt.title("Cosmic hits accumulated over an image cycle", fontsize = 24)
plt.xlabel("x$_{CCD}$ [pixels]", fontsize = 20)
plt.ylabel("y$_{CCD}$ [pixels]", fontsize = 20)

for tick in ax.xaxis.get_major_ticks():
    tick.label.set_fontsize(16)
    
for tick in ax.yaxis.get_major_ticks():
    tick.label.set_fontsize(16)

## Distribution

In [ ]:
# Normal distribution

def model_normal(x, mu, sigma):
    return 1. / (sigma*np.sqrt(2*np.pi)) * np.exp(-1/2*((x-mu)/sigma)**2)

# Model of skewed normal distribution

def phi(x):
    return 1. / np.sqrt(2*np.pi) * np.exp(-x**2 / 2.)

def psi(x):
    return 0.5*(1 + erf(x/np.sqrt(2)))

def model_skewed(x, alpha, omega, xi):
    """
    Skewed normal distribution.
    """
    return 2. / omega * phi((x-xi)/omega) * psi(alpha*(x-xi)/omega)

In [ ]:
# Plot model differences

x = np.linspace(0, 10000, 1000)
n = model_normal(x, 2500, 2000)
s = model_skewed(x, 30, 2000, 2500)
plt.figure(figsize=(8,6))
plt.plot(x, n, '--')
plt.plot(x, s, '-')
plt.xlabel('Electron charge deposited [e-]')
plt.ylabel('Fraction of events')
plt.xlim(0, max(x))
plt.show()